# Hugging Face `transformers` — Lab Notebook

## Module 01 — Setup & First Pipeline

In [ ]:
# Reinstall a specific, known-good version of transformers
!pip uninstall -y transformers
!pip install transformers==4.56.1

### Quick sentiment pipeline (no model specified)

In [1]:
from transformers import pipeline

sentiment_clf = pipeline("sentiment-analysis")

output = sentiment_clf("I love learning AI")
print(output)

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english.
[{'label': 'POSITIVE', 'score': 0.9975565671920776}]


### Same pipeline, but with the model named explicitly

In [ ]:
from transformers import pipeline

sentiment_clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

### Loading a base BERT tokenizer + model

In [1]:
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
bert_model = AutoModel.from_pretrained("bert-base-cased")

Some weights of BertModel were not used when loading from bert-base-cased
(the pretraining-head weights are dropped, which is expected for this task).


### Practical activity — run the classifier on a new sentence

In [1]:
from transformers import pipeline

sentiment_clf = pipeline("sentiment-analysis")

output = sentiment_clf("I'm happy with the service")
print(output)

[{'label': 'POSITIVE', 'score': 0.9998645782470703}]


## Module 02 — Named Models, Generation, and QA

In [1]:
from transformers import pipeline

sentiment_clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

output = sentiment_clf("I love learning Artificial Intelligence")
print(output)

[{'label': 'POSITIVE', 'score': 0.999432384967804}]


### Text generation with GPT-2

In [1]:
from transformers import pipeline

text_gen = pipeline("text-generation", model="gpt2")

generated = text_gen("Artificial Intelligence is", max_new_tokens=50)
print(generated[0]["generated_text"])

Artificial Intelligence is a full featured game system for 3D CAD, with a huge range of possibilities to enhance your gameplay. You can connect your smartphone to the GamePad and play 2D games, or you can play with your computer and get the real-time game


### Checking the installed environment

In [ ]:
import sys
import transformers

print("transformers version:", transformers.__version__)
print("python version:", sys.version)
print("transformers path:", transformers.__file__)

!pip show transformers

### Extractive question answering

In [1]:
from transformers import pipeline

qa_pipeline = pipeline(model="distilbert-base-cased-distilled-squad")

result = qa_pipeline(
    question="What is AI?",
    context="Artificial Intelligence is the simulation of human intelligence by machines.",
)
print(result)

{'score': 0.3638593554496765, 'start': 27, 'end': 75, 'answer': 'the simulation of human intelligence by machines'}


## Module 03 — Tokenizers and Hidden States

In [1]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sample_text = "I love AI"
token_pieces = bert_tokenizer.tokenize(sample_text)
print(token_pieces)

['I', 'love', 'AI']


In [ ]:
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
bert_model = AutoModel.from_pretrained("bert-base-cased")

In [1]:
sample_text = "I love learning AI"

model_inputs = bert_tokenizer(sample_text, return_tensors="pt")
model_outputs = bert_model(**model_inputs)

print(model_outputs.last_hidden_state.shape)

torch.Size([1, 6, 768])


## Module 04 — Generation, QA, Summarization, Translation

In [1]:
text_gen = pipeline("text-generation", model="gpt2")

generated = text_gen("Artificial Intelligence is", max_new_tokens=50)
print(generated[0]["generated_text"])

Artificial Intelligence is a major focus of attention in the field of artificial intelligence. Most of the AI research is coming from organizations such as Google, IBM, and the National Academy of Sciences. Artificial Intelligence, however, will not be the focus of the next 10 years.


In [ ]:
generated = text_gen(
    "The future of AI is",
    max_new_tokens=50,
    temperature=0.7,
)

### Question answering with the default pipeline model

In [1]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering")

passage = """Hugging Face is an AI platform that provides
pretrained models, datasets, and tools."""

query = "What does Hugging Face provide?"

answer = qa_pipeline(question=query, context=passage)
print(answer)

{'score': 0.8230705857276917, 'start': 45, 'end': 83, 'answer': 'pretrained models, datasets, and tools'}


### Summarization

In [1]:
summarizer = pipeline("summarization")

passage = """Artificial Intelligence is
transforming many industries — healthcare,
finance, education, transportation, and more.
Modern AI can analyze large amounts of data
and automate complex tasks."""

summary = summarizer(passage, max_length=50, min_length=20, do_sample=False)
print(summary[0]["summary_text"])

 Artificial Intelligence is transforming many industries — healthcare, finance, education, transportation, and more . Modern AI can analyze large amounts of data and automate complex tasks .


### Translation (English → French)

In [ ]:
translator = pipeline("translation_en_to_fr")

translated = translator("Artificial Intelligence is powerful.")

## Module 05 — Manual Classification Walkthrough (logits → label)

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

clf_model_name = "distilbert-base-uncased-finetuned-sst-2-english"

clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_model_name)

In [1]:
sample_text = "I love learning Artificial Intelligence."

encoded = clf_tokenizer(sample_text, return_tensors="pt")
print(encoded)

{'input_ids': tensor([[ 101, 1045, 2293, 4083, 7976, 4454, 1012,  102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [1]:
raw_output = clf_model(
    input_ids=encoded["input_ids"],
    attention_mask=encoded["attention_mask"],
)
print(raw_output.logits)

tensor([[-3.6871,  3.9370]], grad_fn=<AddmmBackward0>)


In [1]:
class_probs = torch.softmax(raw_output.logits, dim=1)
print(class_probs)

tensor([[4.8829e-04, 9.9951e-01]], grad_fn=<SoftmaxBackward0>)


In [1]:
predicted_idx = torch.argmax(raw_output.logits, dim=1)
print(predicted_idx)

tensor([1])


In [1]:
predicted_label = clf_model.config.id2label[predicted_idx.item()]
print(predicted_label)

POSITIVE


### Same steps, condensed into one block

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

clf_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(clf_model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_model_name)

sample_text = "I love learning Artificial Intelligence."
encoded = clf_tokenizer(sample_text, return_tensors="pt")

raw_output = clf_model(**encoded)

predicted_idx = torch.argmax(raw_output.logits, dim=1)
predicted_label = clf_model.config.id2label[predicted_idx.item()]

print("Text:", sample_text)
print("Prediction:", predicted_label)

Text: I love learning Artificial Intelligence.
Prediction: POSITIVE


In [1]:
class_probs = torch.softmax(raw_output.logits, dim=1)
confidence_score = class_probs[0, predicted_idx]
print(confidence_score.item())

0.99951171875


In [1]:
print("Text:", sample_text)
print("Prediction:", predicted_label)
print("Confidence:", confidence_score)

Text: I love learning Artificial Intelligence.
Prediction: POSITIVE
Confidence: tensor([0.9995], grad_fn=<IndexBackward0>)


### Batch classification over several sentences

In [1]:
batch_texts = [
    "I love AI.",
    "This movie is terrible.",
    "Transformers are amazing!",
]

batch_encoded = clf_tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

batch_output = clf_model(**batch_encoded)
batch_preds = torch.argmax(batch_output.logits, dim=1)
batch_probs = torch.softmax(batch_output.logits, dim=1)

for i, pred in enumerate(batch_preds):
    label = clf_model.config.id2label[pred.item()]
    conf = batch_probs[i, pred].item()
    print(batch_texts[i], "->", label, "->", conf)

I love AI. -> POSITIVE -> 0.9998161196708679
This movie is terrible. -> NEGATIVE -> 0.9997261166572571
Transformers are amazing! -> POSITIVE -> 0.9998725652694702


## Module 07 — Causal Generation, Summarization, Translation, Text2Text

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

gen_model_name = "gpt2"

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name)

prompt_text = "Artificial Intelligence is"
prompt_inputs = gen_tokenizer(prompt_text, return_tensors="pt")

generated_ids = gen_model.generate(**prompt_inputs, max_new_tokens=50)

generated_text = gen_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(generated_text)

Artificial Intelligence is a new field of research that has been gaining traction in recent years. It is a field that has been growing in popularity since the early 1990s.

The field is called Artificial Intelligence and it is a field that has been growing in popularity since


### Summarization with a larger BART model

In [1]:
from transformers import pipeline

bart_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

long_passage = """
Artificial Intelligence is
transforming industries across
the world. It is being used in
healthcare, finance, education,
transportation, and many other
fields. AI helps automate tasks,
analyze large amounts of data,
and improve decision-making.
"""

summary = bart_summarizer(long_passage, max_length=50, min_length=20, do_sample=False)
print(summary[0]["summary_text"])

Artificial Intelligence is transforming industries across the world. It is being used in healthcare, finance, education, and many other fields. AI helps automate tasks, analyze large amounts of data, and improve decision-making.


### Translation with a Helsinki-NLP MarianMT model

In [ ]:
from transformers import pipeline

en_fr_translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")

source_text = "Artificial Intelligence is powerful."
translation = en_fr_translator(source_text)

print(translation)
print(translation[0]["translation_text"])

### FLAN-T5 for text2text tasks (summarization-style prompt)

In [1]:
from transformers import pipeline

flan_pipe = pipeline("text2text-generation", model="google/flan-t5-base")

passage = """
Artificial Intelligence is being used in
healthcare, finance, education, and
transportation.
"""

instruction = "summarize: " + passage

result = flan_pipe(instruction, max_new_tokens=50)
print(result[0]["generated_text"])

Artificial Intelligence is being used in healthcare, finance, education, and transportation.


### FLAN-T5 as a question answerer

In [1]:
qa_prompt = """
question: What is AI?
context: AI is a technology that enables
machines to perform tasks that normally
require human intelligence.
"""

result = flan_pipe(qa_prompt, max_new_tokens=50)
print(result[0]["generated_text"])

a technology that enables machines to perform tasks that normally require human intelligence
